# prepare_batch_forcing notebook

This notebook embeds the full code from `prepare_batch_forcing.py` and runs `main()` directly.

In [ ]:
from pathlib import Path
import os

# Optional setup cell. Main code cell also sets project root.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)
print(f"Working directory: {ROOT}")

In [ ]:
from pathlib import Path
import os

# Notebook: ensure project root (run big cell alone).
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)

"""
Build per-basin rainplusmelt + PET for catchments in basins_selection.csv.

Meteorology: F:\\Data\\estreams_2025\\meteorology\\meteorology\\estreams_meteorology_{basin}.csv
frac_snow: estreams_hydrometeo_signatures.csv
Common period: 1976-01-01 to 2005-12-31 (default)

Outputs (per basin):
  Results/forcing_per_basin/{basin_id}/rainplusmelt.csv  (date, rainplusmelt)
  Results/forcing_per_basin/{basin_id}/PET.csv           (date, pet)

Run pilot (first 10 basins):
  python prepare_batch_forcing.py --limit 10

Parallel:
  python prepare_batch_forcing.py --skip_existing --workers 8
"""

from __future__ import annotations

import argparse
import json
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd

from basin_selection_io import DEFAULT_SELECTION_CSV, list_basin_ids, load_basins_selection
from prepare_eastream_inputs import compute_rainplusmelt_components_for_basin

DEFAULT_METEO_DIR = Path(r"F:\Data\estreams_2025\meteorology\meteorology")
DEFAULT_SIGNATURES = Path("data/csv_estream/estreams_hydrometeo_signatures.csv")
DEFAULT_OUT_DIR = Path("Results/forcing_per_basin")
DEFAULT_PERIOD_START = "1976-01-01"
DEFAULT_PERIOD_END = "2005-12-31"


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawTextHelpFormatter)
    p.add_argument("--selection_csv", type=Path, default=DEFAULT_SELECTION_CSV)
    p.add_argument("--meteo_dir", type=Path, default=DEFAULT_METEO_DIR)
    p.add_argument("--signatures_csv", type=Path, default=DEFAULT_SIGNATURES)
    p.add_argument("--out_dir", type=Path, default=DEFAULT_OUT_DIR)
    p.add_argument("--period_start", default=DEFAULT_PERIOD_START)
    p.add_argument("--period_end", default=DEFAULT_PERIOD_END)
    p.add_argument("--limit", type=int, default=None, help="Process only first N basins (pilot).")
    p.add_argument("--offset", type=int, default=0)
    p.add_argument("--basins", default="", help="Comma-separated basin IDs (overrides limit/offset).")
    p.add_argument("--skip_existing", action="store_true", help="Skip basins with both output CSVs present.")
    p.add_argument("--workers", type=int, default=1, help="Parallel basins (default 1). Try 8 on 10-core PC.")
    return p.parse_known_args()[0]


def _forcing_worker(task: dict) -> dict:
    try:
        meta = build_forcing_one_basin(
            task["basin_id"],
            Path(task["meteo_dir"]),
            task["frac_snow"],
            pd.Timestamp(task["period_start"]),
            pd.Timestamp(task["period_end"]),
            Path(task["out_dir"]),
        )
        return {"status": "ok", **meta}
    except Exception as exc:
        return {"status": "failed", "basin_id": task["basin_id"], "reason": str(exc)}


def load_frac_snow_map(signatures_csv: Path, basin_ids: list[str]) -> dict[str, float]:
    sig = pd.read_csv(signatures_csv, sep=None, engine="python")
    sig.columns = [str(c).strip().lstrip("\ufeff") for c in sig.columns]
    if "basin_id" not in sig.columns:
        unnamed = next((c for c in sig.columns if str(c).lower().startswith("unnamed")), None)
        if unnamed:
            sig = sig.rename(columns={unnamed: "basin_id"})
    sig["basin_id"] = sig["basin_id"].astype(str).str.strip()
    want = set(basin_ids)
    out: dict[str, float] = {}
    for _, row in sig.iterrows():
        bid = row["basin_id"]
        if bid in want:
            out[bid] = float(row["frac_snow"])
    missing = [b for b in basin_ids if b not in out]
    if missing:
        raise KeyError(f"frac_snow missing in signatures for: {missing[:10]} ... ({len(missing)} total)")
    return out


def build_forcing_one_basin(
    basin_id: str,
    meteo_dir: Path,
    frac_snow: float,
    period_start: pd.Timestamp,
    period_end: pd.Timestamp,
    out_dir: Path,
) -> dict:
    meteo_path = meteo_dir / f"estreams_meteorology_{basin_id}.csv"
    if not meteo_path.exists():
        raise FileNotFoundError(f"Missing meteorology file: {meteo_path}")

    df = pd.read_csv(meteo_path)
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    sub = df.loc[(df["date"] >= period_start) & (df["date"] <= period_end)].copy()
    if sub.empty:
        raise ValueError(f"No meteo rows in {period_start.date()}..{period_end.date()} for {basin_id}")

    comp = compute_rainplusmelt_components_for_basin(
        sub[["p_mean", "t_mean"]], frac_snow=frac_snow
    )
    basin_out = out_dir / basin_id
    basin_out.mkdir(parents=True, exist_ok=True)

    rpm_df = pd.DataFrame({"date": sub["date"], "rainplusmelt": comp["rainplusmelt"]})
    pet_df = pd.DataFrame({"date": sub["date"], "pet": sub["pet_mean"].astype(float)})
    rpm_df.to_csv(basin_out / "rainplusmelt.csv", index=False)
    pet_df.to_csv(basin_out / "PET.csv", index=False)

    return {
        "basin_id": basin_id,
        "n_days": len(sub),
        "date_start": str(sub["date"].min().date()),
        "date_end": str(sub["date"].max().date()),
        "frac_snow": frac_snow,
    }


def main() -> None:
    args = parse_args()
    period_start = pd.Timestamp(args.period_start)
    period_end = pd.Timestamp(args.period_end)

    if args.basins.strip():
        basin_ids = [b.strip() for b in args.basins.split(",") if b.strip()]
    else:
        basin_ids = list_basin_ids(args.selection_csv, limit=args.limit, offset=args.offset)

    frac_map = load_frac_snow_map(args.signatures_csv, basin_ids)
    args.out_dir.mkdir(parents=True, exist_ok=True)

    ok: list[dict] = []
    failed: list[dict] = []
    pending: list[str] = []
    for basin_id in basin_ids:
        basin_out = args.out_dir / basin_id
        if args.skip_existing and (basin_out / "rainplusmelt.csv").exists() and (basin_out / "PET.csv").exists():
            continue
        pending.append(basin_id)

    workers = max(1, int(args.workers))
    tasks = [
        {
            "basin_id": basin_id,
            "meteo_dir": str(args.meteo_dir),
            "frac_snow": frac_map[basin_id],
            "period_start": str(period_start.date()),
            "period_end": str(period_end.date()),
            "out_dir": str(args.out_dir),
        }
        for basin_id in pending
    ]

    if workers == 1:
        for i, task in enumerate(tasks, start=1):
            print(f"[{i}/{len(tasks)}] {task['basin_id']} ...")
            out = _forcing_worker(task)
            if out["status"] == "ok":
                ok.append(out)
            else:
                failed.append({"basin_id": out["basin_id"], "stage": "forcing", "reason": out["reason"]})
                print(f"  FAILED: {out['reason']}")
    else:
        print(f"Building forcing for {len(tasks)} basins with {workers} workers ...")
        with ProcessPoolExecutor(max_workers=workers) as pool:
            futures = {pool.submit(_forcing_worker, t): t["basin_id"] for t in tasks}
            done = 0
            for fut in as_completed(futures):
                done += 1
                out = fut.result()
                bid = out.get("basin_id", futures[fut])
                print(f"[{done}/{len(tasks)}] {bid}")
                if out["status"] == "ok":
                    ok.append(out)
                else:
                    failed.append({"basin_id": bid, "stage": "forcing", "reason": out["reason"]})

    summary_dir = args.out_dir / "_batch_logs"
    summary_dir.mkdir(parents=True, exist_ok=True)
    if ok:
        pd.DataFrame(ok).to_csv(summary_dir / "forcing_ok.csv", index=False)
    if failed:
        pd.DataFrame(failed).to_csv(summary_dir / "forcing_failed.csv", index=False)

    manifest = {
        "period_start": str(period_start.date()),
        "period_end": str(period_end.date()),
        "n_requested": len(basin_ids),
        "n_ok": len(ok),
        "n_failed": len(failed),
    }
    (summary_dir / "forcing_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"Done. OK={len(ok)} failed={len(failed)} -> {args.out_dir}")


# Execute the script entry point
main()
